## Import documents exported from Evernote
### Use Ollama which runs locally
#### Prerequisite steps
* Launch Ollama server from terminal: ollama serve
* exported notes from your Evernote notebook as html 
* converted the notes further to md-files and remove broken image links (use python/AI)
* the files are named with note titles

Files are in one folder.

Note: this is built on RAG example by https://github.com/ed-donner/llm_engineering/commits?author=dinorrusso

##### What did I learn?
I thought this to be a trivial task, but it was not 😃
Even though the retriever had the information required, it was dropped from the answer.

Further difficulty for the AI: not all documents are in English.


In [ ]:
# Document loading, retrieval methods and text splitting
!pip install -qU langchain langchain_community

# Local vector store via Chroma
!pip install -qU langchain_chroma

# Local inference and embeddings via Ollama
!pip install -qU langchain_ollama

# Web Loader
!pip install -qU beautifulsoup4

# Pull the model first
!ollama pull nomic-embed-text

!pip install libmagic

In [74]:
#Imports
import os
from dotenv import load_dotenv
import gradio as gr
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores.chroma import Chroma
from langchain.embeddings import OllamaEmbeddings
from langchain_ollama.llms import OllamaLLM


In [ ]:
# Read in documents using LangChain's loaders
# My evernote folder is flat

load_dotenv()

def add_metadata(doc):
    doc_name = os.path.basename(doc.metadata['source'])
    doc.metadata["doc_name"] = doc_name
    return doc

documents = []

folder = os.getenv('EVERNOTE_EXPORT_PATH')
loader = DirectoryLoader(folder, glob="*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
folder_docs = loader.load()
documents.extend([add_metadata(doc) for doc in folder_docs])

text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=10)
chunks = text_splitter.split_documents(documents)

print(f"Total number of chunks: {len(chunks)}")
#print(f"Document names found: {set(doc.metadata['doc_name'] for doc in documents)}")


In [ ]:
# Put the chunks of data into a Vector Store that associates a Vector Embedding with each chunk
# Chroma is a popular open source Vector Database based on SQLLite
db_name = "vector_db"

embeddings = OllamaEmbeddings(model="nomic-embed-text")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

# Create vectorstore

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")


In [88]:
# create a new Chat with Ollama
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain_core.callbacks import StdOutCallbackHandler

# Define a custom prompt template to instruct the LLM
prompt_template = """
You are an AI assistant. 
Please respond to the following query using the information provided in the retrieved documents. 
Do not translate the text and maintain the original language of the documents.

Query: {query}

Retrieved Documents:
{retrieved_documents}

Answer:
"""

MODEL = "llama3.2:latest"
llm = OllamaLLM(temperature=0.7, model=MODEL, prompt_template=prompt_template)

# set up the conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# the retriever is an abstraction over the VectorStore that will be used during RAG
retriever = vectorstore.as_retriever()

# putting it together: set up the conversation chain with the GPT 3.5 LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory, callbacks=[StdOutCallbackHandler()])


In [ ]:
# Let's try a simple question


query = "How to fit the cassette joint to a Shimano Nexus 7 Speed Hub?"
result = conversation_chain.invoke({"question": query})
answer = result["answer"]
print("\nAnswer:", answer)


In [32]:
# set up a new conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# putting it together: set up the conversation chain with the  LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

In [29]:
# Wrapping that in a function

def chat(question, history):
    result = conversation_chain.invoke({"question": question})
    return result["answer"]

## Now we will bring this up in Gradio using the Chat interface -

A quick and easy way to prototype a chat with an LLM

In [ ]:
# And in Gradio:

view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)